# Limpeza e padronizacao - Vacinas (bairro: Coloninha)

Este notebook le o arquivo bruto exportado do sistema G-MUS (`Vacinas_Aplicadas_Coloninha_Anonimizadas.csv`),
padroniza as colunas conforme o layout combinado com a equipe e gera o CSV final por bairro.

**Como usar:**
1. Rode a celula de montagem do Google Drive (ou pule e use upload manual, celula alternativa logo abaixo).
2. Ajuste as variaveis `CAMINHO_ENTRADA` e `CAMINHO_SAIDA` se necessario.
3. Rode as celulas na ordem.


## 1. Montar o Google Drive

Use esta opcao se os arquivos brutos estao na estrutura de pastas do time dentro do Drive
(ex.: `MeuDrive/data/bruto/vacinas/...`).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1b. (Alternativa) Upload manual do arquivo

Se preferir nao usar o Drive, rode esta celula para fazer upload do CSV bruto diretamente
pelo navegador. Depois, ajuste `CAMINHO_ENTRADA` na proxima celula para apontar para o
arquivo enviado (ele fica salvo em `/content/`).

In [ ]:
from google.colab import files

# Descomente as duas linhas abaixo para fazer upload manual em vez de usar o Drive
# uploaded = files.upload()
# print(list(uploaded.keys()))

## 2. Configuracao de caminhos

Ajuste conforme a estrutura de pastas do seu Drive (ou do upload manual).

In [ ]:
from pathlib import Path

BAIRRO = "Coloninha"

# Caminho do arquivo bruto (ajuste para a pasta do seu Drive)
CAMINHO_ENTRADA = Path(
    "/content/drive/MyDrive/Pyanalytics/CSV/Vacinas_Aplicadas_Coloninha_Anonimizadas.csv"
)

# Caminho de saida do CSV padronizado
CAMINHO_SAIDA = Path(
    "/content/drive/MyDrive/Pyanalytics/CSV/coloninha.csv"
)

print("Entrada:", CAMINHO_ENTRADA)
print("Saida:  ", CAMINHO_SAIDA)

Entrada: /content/drive/MyDrive/Pyanalytics/CSV/Vacinas_Aplicadas_Coloninha_Anonimizadas.csv
Saida:   /content/drive/MyDrive/Pyanalytics/CSV/coloninha.csv


## 3. Funcoes de limpeza e padronizacao

O arquivo bruto e um relatorio do sistema G-MUS, com cabecalhos de pagina, blocos por
vacina (`Vacina: <nome> (<codigo>)`) e rodapes que se repetem a cada pagina. As celulas
abaixo identificam apenas as linhas de dado (paciente anonimizado = "XXX") e extraem/
padronizam os campos exigidos.

**Observacoes de qualidade de dado:**
- A maioria das linhas traz a "Data da aplicacao" completa (DD/MM/AAAA).
- Um subconjunto de linhas mais antigas so traz Mes/Ano (MM/AAAA). Para manter o padrao
  DD/MM/AAAA, o dia foi preenchido com "01" como convencao, e a ocorrencia e reportada em
  log para auditoria.
- O nome do bairro nao vem como coluna no arquivo bruto; foi definido manualmente na
  variavel `BAIRRO` acima (extraido do nome do arquivo de origem).

In [ ]:
import csv
import re
import sys

COLUNAS_SAIDA = [
    "Nome do bairro",
    "Nome da vacina",
    "Codigo da vacina",
    "Estrategia",
    "Codigo da estrategia",
    "Dose",
    "Codigo da dose",
    "Vacina estrategia e dose",
    "Lote",
    "Validade do lote",
    "Data da aplicacao",
    "Origem",
]

# "Vacina X (cod) - Estrategia Y (cod) - Dose Z (cod)"
RE_VACINA_ESTRATEGIA_DOSE = re.compile(
    r"^(?P<vacina>.*) \((?P<cod_vacina>\d+)\) - "
    r"(?P<estrategia>.*) \((?P<cod_estrategia>\d+)\) - "
    r"(?P<dose>.*) \((?P<cod_dose>\d+)\)$"
)

# "LOTE - DD/MM/AAAA"
RE_LOTE_VALIDADE = re.compile(r"^(?P<lote>.*) - (?P<validade>\d{2}/\d{2}/\d{4})$")

RE_DATA_COMPLETA = re.compile(r"^\d{2}/\d{2}/\d{4}$")
RE_DATA_MES_ANO = re.compile(r"^(\d{2})/(\d{4})$")


def extrair_linhas_de_dado(linhas):
    """Filtra apenas as linhas de dado de aplicacao (ignora cabecalhos,
    rodapes e linhas de resumo do relatorio)."""
    dados = []
    for linha in linhas:
        # linhas de dado tem 13 ou 14 colunas e a 2a coluna e o paciente
        # anonimizado "XXX"; ha tambem linhas "XXX" vazias (sem vacina
        # aplicada) que devem ser descartadas.
        if len(linha) in (13, 14) and linha[1] == "XXX":
            if len(linha) == 13:
                vac_estr_dose, lote_validade, aplicado, origem = (
                    linha[4], linha[7], linha[8], linha[10]
                )
            else:  # 14 colunas
                vac_estr_dose, lote_validade, aplicado, origem = (
                    linha[4], linha[7], linha[9], linha[11]
                )
            if vac_estr_dose.strip() == "":
                continue
            dados.append((vac_estr_dose, lote_validade, aplicado, origem))
    return dados


def padronizar_data_aplicacao(valor: str) -> str:
    """Garante o formato DD/MM/AAAA. Quando a fonte so traz MM/AAAA
    (registros legados), assume-se o dia 01 como convencao e a ocorrencia
    e reportada via log para auditoria."""
    valor = valor.strip()
    if RE_DATA_COMPLETA.match(valor):
        return valor
    m = RE_DATA_MES_ANO.match(valor)
    if m:
        mes, ano = m.groups()
        print(f"[aviso] Data sem dia informado (\'{valor}\'). Assumindo 01/{mes}/{ano}.",
              file=sys.stderr)
        return f"01/{mes}/{ano}"
    print(f"[aviso] Data em formato inesperado, mantida como veio: \'{valor}\'",
          file=sys.stderr)
    return valor


def processar(caminho_entrada, caminho_saida, bairro):
    with open(caminho_entrada, encoding="utf-8", errors="replace", newline="") as f:
        leitor = csv.reader(f)
        linhas_brutas = list(leitor)

    registros_brutos = extrair_linhas_de_dado(linhas_brutas)

    linhas_saida = []
    for vac_estr_dose, lote_validade, aplicado, origem in registros_brutos:
        m_ved = RE_VACINA_ESTRATEGIA_DOSE.match(vac_estr_dose.strip())
        if not m_ved:
            print(f"[erro] Nao foi possivel interpretar: \'{vac_estr_dose}\'. Linha ignorada.",
                  file=sys.stderr)
            continue

        m_lote = RE_LOTE_VALIDADE.match(lote_validade.strip())
        if m_lote:
            lote = m_lote.group("lote").strip()
            validade = m_lote.group("validade").strip()
        else:
            lote = lote_validade.strip()
            validade = ""
            print(f"[aviso] Lote/validade em formato inesperado: \'{lote_validade}\'",
                  file=sys.stderr)

        data_aplicacao = padronizar_data_aplicacao(aplicado)

        linhas_saida.append({
            "Nome do bairro": bairro,
            "Nome da vacina": m_ved.group("vacina").strip(),
            "Codigo da vacina": m_ved.group("cod_vacina"),
            "Estrategia": m_ved.group("estrategia").strip(),
            "Codigo da estrategia": m_ved.group("cod_estrategia"),
            "Dose": m_ved.group("dose").strip(),
            "Codigo da dose": m_ved.group("cod_dose"),
            "Vacina estrategia e dose": vac_estr_dose.strip(),
            "Lote": lote,
            "Validade do lote": validade,
            "Data da aplicacao": data_aplicacao,
            "Origem": origem.strip(),
        })

    caminho_saida.parent.mkdir(parents=True, exist_ok=True)
    with open(caminho_saida, "w", encoding="utf-8", newline="") as f:
        escritor = csv.DictWriter(f, fieldnames=COLUNAS_SAIDA)
        escritor.writeheader()
        escritor.writerows(linhas_saida)

    print(f"OK: {len(linhas_saida)} registros escritos em {caminho_saida}")
    return linhas_saida

## 4. Executar o processamento

In [ ]:
registros = processar(CAMINHO_ENTRADA, CAMINHO_SAIDA, BAIRRO)
registros[:3]  # espiada nas 3 primeiras linhas geradas

OK: 3012 registros escritos em /content/drive/MyDrive/Pyanalytics/CSV/coloninha.csv


[aviso] Data sem dia informado ('02/2026'). Assumindo 01/02/2026.
[aviso] Data sem dia informado ('03/2026'). Assumindo 01/03/2026.
[aviso] Data sem dia informado ('03/2026'). Assumindo 01/03/2026.
[aviso] Data sem dia informado ('03/2026'). Assumindo 01/03/2026.
[aviso] Data sem dia informado ('03/2026'). Assumindo 01/03/2026.
[aviso] Data sem dia informado ('03/2026'). Assumindo 01/03/2026.
[aviso] Data sem dia informado ('03/2026'). Assumindo 01/03/2026.
[aviso] Data sem dia informado ('03/2026'). Assumindo 01/03/2026.
[aviso] Data sem dia informado ('03/2026'). Assumindo 01/03/2026.
[aviso] Data sem dia informado ('03/2026'). Assumindo 01/03/2026.
[aviso] Data sem dia informado ('03/2026'). Assumindo 01/03/2026.
[aviso] Data sem dia informado ('03/2026'). Assumindo 01/03/2026.
[aviso] Data sem dia informado ('03/2026'). Assumindo 01/03/2026.
[aviso] Data sem dia informado ('03/2026'). Assumindo 01/03/2026.
[aviso] Data sem dia informado ('04/2026'). Assumindo 01/04/2026.
[aviso] Da

[{'Nome do bairro': 'Coloninha',
  'Nome da vacina': 'Vacina raiva em cultivo celular vero',
  'Codigo da vacina': '37',
  'Estrategia': 'Pós-exposição',
  'Codigo da estrategia': '13',
  'Dose': '2ª DOSE',
  'Codigo da dose': '2',
  'Vacina estrategia e dose': 'Vacina raiva em cultivo celular vero (37) - Pós-exposição (13) - 2ª DOSE (2)',
  'Lote': 'Y1B79',
  'Validade do lote': '30/04/2027',
  'Data da aplicacao': '26/03/2026',
  'Origem': 'Prontuário'},
 {'Nome do bairro': 'Coloninha',
  'Nome da vacina': 'Vacina raiva em cultivo celular vero',
  'Codigo da vacina': '37',
  'Estrategia': 'Pós-exposição',
  'Codigo da estrategia': '13',
  'Dose': '3ª DOSE',
  'Codigo da dose': '3',
  'Vacina estrategia e dose': 'Vacina raiva em cultivo celular vero (37) - Pós-exposição (13) - 3ª DOSE (3)',
  'Lote': 'Y1B79',
  'Validade do lote': '30/04/2027',
  'Data da aplicacao': '30/03/2026',
  'Origem': 'Prontuário'},
 {'Nome do bairro': 'Coloninha',
  'Nome da vacina': 'Vacina raiva em cultivo 

## 5. (Opcional) Baixar o CSV gerado

Use esta celula se quiser baixar o arquivo direto para o seu computador
em vez de (ou alem de) salvar no Drive.

In [ ]:
from google.colab import files
files.download(str(CAMINHO_SAIDA))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>